In [1]:
from datasets import Dataset
from transformers import (
 AutoTokenizer,
 AutoModelForSequenceClassification,
 TrainingArguments,
 Trainer,
)

/home/saleh/Projects/multi-agent-teq-support/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# MODEL A

In [2]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_A = 'distilbert-base-uncased'
LABELS = [
    'authentication',
    'network',
    'deployment',
    'database',
    'gpu',
    'api',
    'package',
    'general',
]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

tokenizer_a = AutoTokenizer.from_pretrained(MODEL_A)
model_a = AutoModelForSequenceClassification.from_pretrained(
    MODEL_A,
    num_labels=len(LABELS),
    label2id=label2id,
    id2label=id2label
    )

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1067.74it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
def tokenize_a(batch):
    return tokenizer_a(batch['text'], truncation=True, max_length=128)

In [4]:
def compute_cls_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision_macro': p,
        'recall_macro': r,
        'f1_macro': f1,
    }

In [5]:
data = [
    # =========================
    # authentication (20)
    # =========================
    {"text": "I cannot log into my account", "label": "authentication"},
    {"text": "My password is not working", "label": "authentication"},
    {"text": "The login page keeps rejecting my credentials", "label": "authentication"},
    {"text": "I forgot my password and cannot sign in", "label": "authentication"},
    {"text": "My authentication token has expired", "label": "authentication"},
    {"text": "The access token is being rejected", "label": "authentication"},
    {"text": "I keep getting unauthorized when logging in", "label": "authentication"},
    {"text": "Two factor authentication is not working", "label": "authentication"},
    {"text": "The verification code never arrives", "label": "authentication"},
    {"text": "My account is locked after several login attempts", "label": "authentication"},
    {"text": "How do I reset my login password?", "label": "authentication"},
    {"text": "The OAuth login is failing", "label": "authentication"},
    {"text": "My session keeps expiring immediately", "label": "authentication"},
    {"text": "I get a 401 error when trying to sign in", "label": "authentication"},
    {"text": "The authentication credentials are invalid", "label": "authentication"},
    {"text": "I cannot refresh my access token", "label": "authentication"},
    {"text": "Login works for other users but not for me", "label": "authentication"},
    {"text": "The authentication service rejects my token", "label": "authentication"},
    {"text": "I am stuck on the account verification step", "label": "authentication"},
    {"text": "Why does my application keep asking me to log in?", "label": "authentication"},

    # =========================
    # network (20)
    # =========================
    {"text": "The server cannot connect to the internet", "label": "network"},
    {"text": "My application cannot reach the remote server", "label": "network"},
    {"text": "The network connection keeps timing out", "label": "network"},
    {"text": "I am getting connection refused errors", "label": "network"},
    {"text": "The service cannot connect to the database server", "label": "network"},
    {"text": "Requests to the server are timing out", "label": "network"},
    {"text": "The API is unreachable from my machine", "label": "network"},
    {"text": "DNS resolution is failing", "label": "network"},
    {"text": "My application cannot resolve the hostname", "label": "network"},
    {"text": "The connection drops randomly", "label": "network"},
    {"text": "I cannot connect to the service over HTTPS", "label": "network"},
    {"text": "The server is unreachable from the container", "label": "network"},
    {"text": "Network requests are extremely slow", "label": "network"},
    {"text": "The connection times out after 30 seconds", "label": "network"},
    {"text": "My Docker container cannot access the internet", "label": "network"},
    {"text": "The application cannot connect to the remote host", "label": "network"},
    {"text": "I am getting a network unreachable error", "label": "network"},
    {"text": "The proxy connection is failing", "label": "network"},
    {"text": "The service works locally but not over the network", "label": "network"},
    {"text": "Why can my machine not reach the server?", "label": "network"},

    # =========================
    # deployment (20)
    # =========================
    {"text": "My application fails when I deploy it", "label": "deployment"},
    {"text": "The deployment keeps failing", "label": "deployment"},
    {"text": "The application works locally but fails in production", "label": "deployment"},
    {"text": "How do I deploy this application?", "label": "deployment"},
    {"text": "The production deployment is stuck", "label": "deployment"},
    {"text": "My deployment pipeline failed", "label": "deployment"},
    {"text": "The new version was not deployed", "label": "deployment"},
    {"text": "The deployment process stops during the build", "label": "deployment"},
    {"text": "The application crashes after deployment", "label": "deployment"},
    {"text": "The production server is still running the old version", "label": "deployment"},
    {"text": "The deployment job keeps getting cancelled", "label": "deployment"},
    {"text": "How can I roll back my deployment?", "label": "deployment"},
    {"text": "The container fails to start after deployment", "label": "deployment"},
    {"text": "My CI deployment is not triggering", "label": "deployment"},
    {"text": "The deployment succeeded but the application is unavailable", "label": "deployment"},
    {"text": "I get an error while deploying to production", "label": "deployment"},
    {"text": "The release pipeline is stuck", "label": "deployment"},
    {"text": "The application is missing after deployment", "label": "deployment"},
    {"text": "The deployment environment has incorrect settings", "label": "deployment"},
    {"text": "Why does my deployment work locally but not in production?", "label": "deployment"},

    # =========================
    # database (20)
    # =========================
    {"text": "I cannot connect to my database", "label": "database"},
    {"text": "The database connection is failing", "label": "database"},
    {"text": "My SQL query returns an error", "label": "database"},
    {"text": "The database is running out of connections", "label": "database"},
    {"text": "How do I create a database table?", "label": "database"},
    {"text": "My database queries are very slow", "label": "database"},
    {"text": "The application cannot find the database", "label": "database"},
    {"text": "I am getting a database timeout", "label": "database"},
    {"text": "The database credentials are not accepted", "label": "database"},
    {"text": "My table data disappeared", "label": "database"},
    {"text": "The migration failed on the database", "label": "database"},
    {"text": "How do I run a database migration?", "label": "database"},
    {"text": "The database schema is out of date", "label": "database"},
    {"text": "I get a duplicate key error from the database", "label": "database"},
    {"text": "The database server is not responding", "label": "database"},
    {"text": "My application cannot execute SQL queries", "label": "database"},
    {"text": "How can I back up the database?", "label": "database"},
    {"text": "The database transaction keeps failing", "label": "database"},
    {"text": "I cannot insert records into the table", "label": "database"},
    {"text": "Why is my database query taking so long?", "label": "database"},

    # =========================
    # gpu (20)
    # =========================
    {"text": "PyTorch cannot detect my GPU", "label": "gpu"},
    {"text": "CUDA is not detecting my graphics card", "label": "gpu"},
    {"text": "My GPU is not being used during training", "label": "gpu"},
    {"text": "I am getting a CUDA out of memory error", "label": "gpu"},
    {"text": "The model training is running on the CPU instead of the GPU", "label": "gpu"},
    {"text": "How do I check if CUDA is available?", "label": "gpu"},
    {"text": "My GPU memory is full", "label": "gpu"},
    {"text": "CUDA initialization failed", "label": "gpu"},
    {"text": "The NVIDIA driver is not detected", "label": "gpu"},
    {"text": "TensorFlow cannot find my GPU", "label": "gpu"},
    {"text": "Why is GPU utilization at zero?", "label": "gpu"},
    {"text": "My CUDA version is incompatible with PyTorch", "label": "gpu"},
    {"text": "The GPU crashes when I start training", "label": "gpu"},
    {"text": "How can I move my model to the GPU?", "label": "gpu"},
    {"text": "I keep getting CUDA memory errors", "label": "gpu"},
    {"text": "The GPU is available but training does not use it", "label": "gpu"},
    {"text": "My graphics card is not visible inside Docker", "label": "gpu"},
    {"text": "CUDA cannot initialize the GPU", "label": "gpu"},
    {"text": "The model runs out of VRAM during inference", "label": "gpu"},
    {"text": "Why does my machine detect the GPU but PyTorch does not?", "label": "gpu"},

    # =========================
    # api (20)
    # =========================
    {"text": "The API returns a 500 error", "label": "api"},
    {"text": "How do I call this API?", "label": "api"},
    {"text": "My API request is failing", "label": "api"},
    {"text": "The endpoint returns invalid JSON", "label": "api"},
    {"text": "I cannot send a POST request to the API", "label": "api"},
    {"text": "The API response is missing a field", "label": "api"},
    {"text": "How do I authenticate an API request?", "label": "api"},
    {"text": "The API endpoint returns 404", "label": "api"},
    {"text": "My GET request returns an unexpected response", "label": "api"},
    {"text": "The API is returning a 429 error", "label": "api"},
    {"text": "How can I pass parameters to the API?", "label": "api"},
    {"text": "The API request body is rejected", "label": "api"},
    {"text": "I cannot connect to the API endpoint", "label": "api"},
    {"text": "The API returns an empty response", "label": "api"},
    {"text": "How do I create an API key?", "label": "api"},
    {"text": "My API calls suddenly started failing", "label": "api"},
    {"text": "The endpoint is returning the wrong status code", "label": "api"},
    {"text": "How do I send headers with the API request?", "label": "api"},
    {"text": "The API rate limit is being exceeded", "label": "api"},
    {"text": "Why does this API request return an error?", "label": "api"},

    # =========================
    # package (20)
    # =========================
    {"text": "I cannot install the transformers package", "label": "package"},
    {"text": "Pip says the package could not be found", "label": "package"},
    {"text": "My Python package installation failed", "label": "package"},
    {"text": "There is a dependency conflict between packages", "label": "package"},
    {"text": "How do I install this Python library?", "label": "package"},
    {"text": "The package version is incompatible", "label": "package"},
    {"text": "Pip cannot resolve my dependencies", "label": "package"},
    {"text": "I get ModuleNotFoundError after installing the package", "label": "package"},
    {"text": "How do I upgrade a Python package?", "label": "package"},
    {"text": "The package is not available for my Python version", "label": "package"},
    {"text": "My requirements file fails to install", "label": "package"},
    {"text": "Pip installed the package but Python cannot import it", "label": "package"},
    {"text": "How do I uninstall a Python package?", "label": "package"},
    {"text": "The package requires a different version of Python", "label": "package"},
    {"text": "I am getting an error while running pip install", "label": "package"},
    {"text": "Two packages require incompatible versions", "label": "package"},
    {"text": "How can I create a Python virtual environment?", "label": "package"},
    {"text": "My dependency installation is stuck", "label": "package"},
    {"text": "The package manager cannot resolve the dependencies", "label": "package"},
    {"text": "Why can I install the package but not import it?", "label": "package"},

    # =========================
    # general (20)
    # =========================
    {"text": "How do I get started with the platform?", "label": "general"},
    {"text": "Where can I find the documentation?", "label": "general"},
    {"text": "Can you explain how this system works?", "label": "general"},
    {"text": "What features does the platform provide?", "label": "general"},
    {"text": "Where can I find an example project?", "label": "general"},
    {"text": "Is there a beginner guide available?", "label": "general"},
    {"text": "How can I learn more about the platform?", "label": "general"},
    {"text": "Where can I find the configuration documentation?", "label": "general"},
    {"text": "What are the system requirements?", "label": "general"},
    {"text": "Can you explain the basic workflow?", "label": "general"},
    {"text": "Where can I find the user guide?", "label": "general"},
    {"text": "Is there an example of how to use this system?", "label": "general"},
    {"text": "What does this service do?", "label": "general"},
    {"text": "How can I configure the application?", "label": "general"},
    {"text": "Where can I find troubleshooting information?", "label": "general"},
    {"text": "Is there a quick start tutorial?", "label": "general"},
    {"text": "What is the recommended way to use this platform?", "label": "general"},
    {"text": "Where can I find more information about this feature?", "label": "general"},
    {"text": "Can you give me an overview of the system?", "label": "general"},
    {"text": "I need general help using the application", "label": "general"},
]

In [6]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset = dataset.map(
    lambda x: {'label': label2id[x['label']]}
)


Map: 100%|██████████| 160/160 [00:00<00:00, 30762.72 examples/s]


In [7]:
from collections import Counter
print(Counter(dataset['label']))

Counter({0: 20, 1: 20, 2: 20, 3: 20, 4: 20, 5: 20, 6: 20, 7: 20})


In [8]:
data_list = dataset.to_list()

texts = [example['text'] for example in data_list]
labels = [example['label'] for example in data_list]

In [9]:
from sklearn.model_selection import train_test_split

train_a, temp_a = train_test_split(
    data_list,
    test_size=0.3,
    stratify=[example['label'] for example in data_list],
    random_state=42
)

val_a, test_a = train_test_split(
temp_a,
test_size=0.5,
stratify=[example['label'] for example in temp_a],
random_state=42
)

In [10]:
from datasets import Dataset

train_a = Dataset.from_list(train_a)
val_a = Dataset.from_list(val_a)
test_a = Dataset.from_list(test_a)

In [11]:
train_a = train_a.map(tokenize_a, batched=True)
val_a = val_a.map(tokenize_a, batched=True)
test_a = test_a.map(tokenize_a, batched=True)

Map: 100%|██████████| 24/24 [00:00<00:00, 7437.26 examples/s]


In [12]:
train_a = train_a.remove_columns(['text'])
val_a = val_a.remove_columns(['text'])
test_a = test_a.remove_columns(['text'])

In [13]:
print(train_a)

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 112
})


In [14]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer_a)

args_a = TrainingArguments(
    output_dir='models/intent_classifier',
    learning_rate=2e-5,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    num_train_epochs=15,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    report_to='none',
    logging_steps=-1
)

trainer_a = Trainer(
    model=model_a,
    args=args_a,
    train_dataset=train_a,
    eval_dataset=val_a,
    compute_metrics=compute_cls_metrics,
    data_collator=data_collator
)

In [15]:
trainer_a.train()
trainer_a.evaluate(test_a)
trainer_a.save_model("models/intent_classifier")
tokenizer_a.save_pretrained("models/intent_classifier")

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,No log,2.055832,0.250000,0.268750,0.250000,0.195109
2,No log,1.979589,0.416667,0.564394,0.416667,0.423810
3,No log,1.813780,0.583333,0.612500,0.583333,0.560317
4,No log,1.591847,0.791667,0.887500,0.791667,0.789583
5,No log,1.371192,0.833333,0.900000,0.833333,0.825000
6,No log,1.165085,0.958333,0.968750,0.958333,0.957143
7,No log,1.014046,0.916667,0.950000,0.916667,0.918750
8,No log,0.877168,0.916667,0.950000,0.916667,0.918750
9,No log,0.778046,0.916667,0.950000,0.916667,0.918750
10,No log,0.708212,0.916667,0.950000,0.916667,0.918750


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.44it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
0.996798,1.096373,15,0.958333,0.968750,0.958333,0.957143


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.45it/s]


('models/intent_classifier/tokenizer_config.json',
 'models/intent_classifier/tokenizer.json')

In [16]:
from transformers import AutoModelForQuestionAnswering

MODEL_B = "distilbert-base-uncased-distilled-squad"
tokenizer_b = AutoTokenizer.from_pretrained(MODEL_B)
model_b = AutoModelForQuestionAnswering.from_pretrained(MODEL_B)

MAX_LENGTH = 384
DOC_STRIDE = 96

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1171.34it/s]


# MODEL B

In [17]:
raw_qa_data = [
    # ===== authentication =====
    {
        "context": "To sign in, users must provide their email and password on the login page. Multi-factor authentication is required for all accounts created after January 2024, and users receive a verification code via SMS. Access tokens are issued using OAuth2 and expire after 60 minutes. If a token expires, the client must call the /auth/refresh endpoint using the refresh token to obtain a new access token. Accounts are automatically locked for 15 minutes after 5 failed login attempts.",
        "qas": [
            ("How long do access tokens last before expiring?", "60 minutes"),
            ("What endpoint should be called to obtain a new access token?", "/auth/refresh"),
            ("How many failed login attempts trigger an account lock?", "5"),
            ("How is the verification code delivered to users?", "SMS"),
            ("What protocol is used to issue access tokens?", "OAuth2"),
            ("What must users provide on the login page?", "email and password"),
            ("When is multi-factor authentication required from?", "January 2024"),
            ("How long are accounts locked after failed attempts?", "15 minutes"),
            ("What must be used to obtain a new access token?", "refresh token"),
            ("What page do users use to sign in?", "login page"),
        ],
    },
    # ===== network =====
    {
        "context": "All internal services communicate over a private VPC using HTTPS on port 443. The API gateway enforces a request timeout of 30 seconds for any upstream call. DNS resolution for internal services is handled by CoreDNS running inside the Kubernetes cluster. If a service cannot reach the database, check whether the network policy allows egress traffic on port 5432. The load balancer health check interval is set to 10 seconds.",
        "qas": [
            ("What port do internal services use for HTTPS communication?", "443"),
            ("What is the request timeout enforced by the API gateway?", "30 seconds"),
            ("What handles DNS resolution for internal services?", "CoreDNS"),
            ("What port must network policy allow for database egress traffic?", "5432"),
            ("How often does the load balancer perform health checks?", "10 seconds"),
            ("What type of network do internal services communicate over?", "private VPC"),
            ("What protocol is used for internal service communication?", "HTTPS"),
            ("Where does CoreDNS run?", "Kubernetes cluster"),
            ("What should be checked if a service cannot reach the database?", "network policy"),
            ("What kind of traffic must the network policy allow on port 5432?", "egress traffic"),
        ],
    },
    # ===== deployment =====
    {
        "context": "Deployments are managed through a CI/CD pipeline that builds a Docker image and pushes it to the container registry on every merge to the main branch. The production environment requires manual approval before a deployment can proceed. Once approved, Dokploy pulls the latest image and performs a rolling update with zero downtime. Rollbacks can be triggered by redeploying the previous image tag stored in the registry. The deployment health check waits up to 90 seconds for the new container to report healthy.",
        "qas": [
            ("What tool performs the rolling update in production?", "Dokploy"),
            ("What branch triggers the CI/CD pipeline to build a new image?", "main branch"),
            ("Where is the Docker image pushed to?", "container registry"),
            ("How long does the deployment health check wait for the container to report healthy?", "90 seconds"),
            ("What kind of update does Dokploy perform?", "rolling update"),
            ("What must happen before a deployment can proceed in production?", "manual approval"),
            ("What does Dokploy pull to perform the update?", "latest image"),
            ("How can rollbacks be triggered?", "redeploying the previous image tag"),
            ("What kind of downtime does the rolling update have?", "zero downtime"),
            ("What builds the Docker image?", "CI/CD pipeline"),
        ],
    },
    # ===== database =====
    {
        "context": "The production database runs PostgreSQL 16 and is configured with a maximum connection pool size of 100. Automated backups run every night at 2 AM and are retained for 30 days. Read replicas are used to offload reporting queries from the primary instance. If the connection pool is exhausted, new requests will fail with a pool_timeout error after waiting 10 seconds. Database migrations must be applied using the migrate command before deploying a new application version.",
        "qas": [
            ("What version of PostgreSQL does the production database run?", "PostgreSQL 16"),
            ("What is the maximum connection pool size?", "100"),
            ("How long are automated backups retained?", "30 days"),
            ("What error occurs when the connection pool is exhausted?", "pool_timeout"),
            ("What command must be used to apply database migrations?", "migrate"),
            ("What time do automated backups run?", "2 AM"),
            ("What are read replicas used for?", "offload reporting queries"),
            ("How long do new requests wait before failing with a pool_timeout error?", "10 seconds"),
            ("What instance do read replicas offload reporting queries from?", "primary instance"),
            ("When must migrations be applied?", "before deploying a new application version"),
        ],
    },
    # ===== gpu =====
    {
        "context": "Training jobs require CUDA 12.1 and a compatible NVIDIA driver version of at least 535. GPU memory usage can be monitored using the nvidia-smi command. When CUDA is unavailable, PyTorch automatically falls back to CPU execution, which significantly increases training time. QLoRA training requires the bitsandbytes library and reduces GPU memory usage by loading the base model in 4-bit precision. Out of memory errors can often be resolved by reducing the per_device_train_batch_size.",
        "qas": [
            ("What CUDA version is required for training jobs?", "CUDA 12.1"),
            ("What library does QLoRA training require?", "bitsandbytes"),
            ("What precision does QLoRA use to load the base model?", "4-bit"),
            ("What parameter can be reduced to resolve out of memory errors?", "per_device_train_batch_size"),
            ("What can nvidia-smi be used to monitor?", "GPU memory usage"),
            ("What is the minimum required NVIDIA driver version?", "535"),
            ("What does PyTorch fall back to when CUDA is unavailable?", "CPU execution"),
            ("What kind of errors can be resolved by reducing per_device_train_batch_size?", "Out of memory errors"),
            ("What must be compatible with CUDA 12.1 for training jobs?", "NVIDIA driver version"),
            ("What command monitors GPU memory usage?", "nvidia-smi"),
        ],
    },
    # ===== api =====
    {
        "context": "The support agent exposes an OpenAI-compatible endpoint at /v1/chat/completions, which accepts POST requests only. Every request must include a valid Bearer token in the Authorization header. Rate limiting is enforced at 60 requests per minute per API key. If the request payload exceeds 1 megabyte, the server responds with a 413 status code. Streaming responses are not supported in the current version of the API.",
        "qas": [
            ("What endpoint does the support agent expose?", "/v1/chat/completions"),
            ("What HTTP method does the endpoint accept?", "POST"),
            ("What is the rate limit per API key?", "60 requests per minute"),
            ("What status code is returned when the payload exceeds 1 megabyte?", "413"),
            ("What header must contain a valid Bearer token?", "Authorization"),
            ("What token type must be included in the Authorization header?", "Bearer token"),
            ("What is the maximum payload size before the server responds with 413?", "1 megabyte"),
            ("Is streaming supported in the current version of the API?", "not supported"),
            ("What compatibility standard does the endpoint follow?", "OpenAI-compatible"),
            ("What must every request include?", "valid Bearer token"),
        ],
    },
    # ===== package =====
    {
        "context": "The project depends on transformers version 4.44 or higher for compatibility with the SFTTrainer class. Installing bitsandbytes on Windows requires a precompiled wheel, since the official package only supports Linux. Dependency conflicts between accelerate and transformers can usually be resolved by upgrading both packages to their latest compatible versions. The peft library must be installed separately to enable LoRA and QLoRA fine-tuning. Virtual environments should be created using Python 3.11 for full compatibility with this project.",
        "qas": [
            ("What version of transformers is required for SFTTrainer compatibility?", "4.44"),
            ("What operating system does the official bitsandbytes package support?", "Linux"),
            ("What library must be installed separately to enable LoRA and QLoRA?", "peft"),
            ("What Python version should be used for virtual environments?", "Python 3.11"),
            ("What class requires transformers 4.44 or higher?", "SFTTrainer"),
            ("What does installing bitsandbytes on Windows require?", "precompiled wheel"),
            ("What two packages can have dependency conflicts?", "accelerate and transformers"),
            ("How can dependency conflicts between accelerate and transformers be resolved?", "upgrading both packages to their latest compatible versions"),
            ("What fine-tuning methods does peft enable?", "LoRA and QLoRA"),
            ("What must be created using Python 3.11?", "Virtual environments"),
        ],
    },
    # ===== general =====
    {
        "context": "This platform is a multi-model agentic technical support system that combines three fine-tuned specialists with a routing layer and a set of domain tools. New users should start by reading the README, which explains the architecture and setup steps. The system is deployed using Docker Compose and can be accessed through Open WebUI once the containers are running. Observability is provided through Langfuse, which traces every request from the router decision to the final response. Support for the platform is available through the internal documentation search tool.",
        "qas": [
            ("How many fine-tuned specialists does the system combine?", "three"),
            ("What should new users read first?", "README"),
            ("What tool provides observability for the system?", "Langfuse"),
            ("What is used to deploy the system?", "Docker Compose"),
            ("What can users access once containers are running?", "Open WebUI"),
            ("What does the README explain?", "architecture and setup steps"),
            ("What does the system combine besides three specialists?", "routing layer and a set of domain tools"),
            ("What does Langfuse trace?", "the router decision to the final response"),
            ("How is support for the platform available?", "internal documentation search tool"),
            ("What kind of system is this platform?", "multi-model agentic technical support system"),
        ],
    },
]

# Flatten into SQuAD-style records with computed answer_start (no manual counting)
qa_dataset = []
for entry in raw_qa_data:
    context = entry["context"]
    for question, answer_text in entry["qas"]:
        start = context.find(answer_text)
        if start == -1:
            raise ValueError(f"Answer text not found in context: {answer_text!r}")
        qa_dataset.append({
            "question": question,
            "context": context,
            "answer_text": answer_text,
            "answer_start": start,
        })

print(f"Total QA pairs: {len(qa_dataset)}")

Total QA pairs: 80


In [18]:
train_b, temp_b = train_test_split(
    qa_dataset,
    test_size=0.3,
    random_state=42,
)

val_b, test_b = train_test_split(
    temp_b,
    test_size=0.5,
    random_state=42
)

print(len(train_b), len(val_b), len(test_b))

56 12 12


In [19]:
def prepare_qa_features(examples):
    tokenized = tokenizer_b(
        [q.strip() for q in examples['question']],
        examples['context'],
        truncation='only_second',
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length'
    )

    sample_mapping = tokenized.pop('overflow_to_sample_mapping')
    offsets = tokenized.pop('offset_mapping')
    start_positions, end_positions = [], []

    for feature_index, feature_offsets in enumerate(offsets):
        input_ids = tokenized['input_ids'][feature_index]
        cls_index = input_ids.index(tokenizer_b.cls_token_id)
        sequence_ids = tokenized.sequence_ids(feature_index)
        sample_index = sample_mapping[feature_index]

        answer_start = examples['answer_start'][sample_index]
        answer_end = answer_start + len(examples['answer_text'][sample_index])

        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1
        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1

        if (
            feature_offsets[context_start][0] > answer_start
            or feature_offsets[context_end][1] < answer_end
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        token_start = context_start
        while feature_offsets[token_start][1] <= answer_start:
            token_start += 1

        token_end = context_end
        while feature_offsets[token_end][0] >= answer_end:
            token_end -= 1

        start_positions.append(token_start)
        end_positions.append(token_end)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions'] = end_positions
    return tokenized

In [20]:
from datasets import Dataset

train_b = Dataset.from_list(train_b)
val_b = Dataset.from_list(val_b)
test_b = Dataset.from_list(test_b)

qa_train_features = train_b.map(prepare_qa_features, batched=True, remove_columns=train_b.column_names)
qa_val_features = val_b.map(prepare_qa_features, batched=True, remove_columns=val_b.column_names)

Map: 100%|██████████| 12/12 [00:00<00:00, 2247.85 examples/s]


In [21]:
args_b = TrainingArguments(
    output_dir='models/intent_classifier',
    learning_rate=2e-5,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    num_train_epochs=15,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    logging_steps=-1
)

trainer_b = Trainer(
    model=model_b,
    args=args_b,
    train_dataset=qa_train_features,
    eval_dataset=qa_val_features,
    data_collator=data_collator
)

In [22]:
trainer_b.train()
trainer_b.save_model('models/qa_model')
tokenizer_b.save_pretrained('models/qa_model')

Epoch,Training Loss,Validation Loss
1,No log,0.300348


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.73it/s]


KeyboardInterrupt: 

In [ ]:
test_b_tokenized = tokenizer_b(
    [ex["question"].strip() for ex in test_b],
    [ex["context"] for ex in test_b],
    truncation="only_second",
    max_length=MAX_LENGTH,
    stride=DOC_STRIDE,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding="max_length",
)

print(len(test_b_tokenized["input_ids"]))

12


In [ ]:
import torch

model_b.eval()

all_start_logits = []
all_end_logits = []

device = next(model_b.parameters()).device
print(device)  # confirm it's mps

with torch.no_grad():
    for i in range(len(test_b_tokenized["input_ids"])):
        input_ids = torch.tensor([test_b_tokenized["input_ids"][i]]).to(device)
        attention_mask = torch.tensor([test_b_tokenized["attention_mask"][i]]).to(device)
        outputs = model_b(input_ids=input_ids, attention_mask=attention_mask)
        all_start_logits.append(outputs.start_logits[0].cpu())
        all_end_logits.append(outputs.end_logits[0].cpu())

print(len(all_start_logits), all_start_logits[0].shape)

cuda:0
12 torch.Size([384])


In [ ]:
def get_predicted_answer(feature_index, start_logits, end_logits, offsets, context):
    start_idx = int(torch.argmax(start_logits))
    end_idx = int(torch.argmax(end_logits))

    if start_idx > end_idx or start_idx >= len(offsets) or end_idx >= len(offsets):
        return ""

    start_char = offsets[start_idx][0]
    end_char = offsets[end_idx][1]

    if start_char == 0 and end_char == 0:
        return ""  # points to [CLS]/padding, meaning "no answer found"

    return context[start_char:end_char]

In [ ]:
sample_mapping = test_b_tokenized["overflow_to_sample_mapping"]
offsets = test_b_tokenized["offset_mapping"]

predicted_answers = {}

for feature_index in range(len(all_start_logits)):
    sample_index = sample_mapping[feature_index]
    context = test_b[sample_index]["context"]

    pred_text = get_predicted_answer(
        feature_index,
        all_start_logits[feature_index],
        all_end_logits[feature_index],
        offsets[feature_index],
        context,
    )

    # If a sample was split into multiple features, keep the first non-empty prediction
    if sample_index not in predicted_answers or predicted_answers[sample_index] == "":
        predicted_answers[sample_index] = pred_text

print(list(predicted_answers.items())[:5])

[(0, 'OAuth2'), (1, 'Docker Compose'), (2, '16'), (3, 'valid Bearer token'), (4, 'pool_timeout error')]


In [ ]:
import re

def normalize(s):
    return re.sub(r"[^\w\s]", "", s.lower()).strip()

em_count = 0
f1_total = 0

for i, example in enumerate(test_b):
    pred = predicted_answers.get(i, "")
    truth = example["answer_text"]

    pred_norm = normalize(pred)
    truth_norm = normalize(truth)

    # exact match
    if pred_norm == truth_norm:
        em_count += 1

    # token f1
    pred_words = pred_norm.split()
    truth_words = truth_norm.split()
    common = sum(1 for w in pred_words if w in truth_words)

    if len(pred_words) == 0 or len(truth_words) == 0:
        f1 = 0
    else:
        precision = common / len(pred_words)
        recall = common / len(truth_words)
        f1 = 0 if (precision + recall == 0) else 2 * precision * recall / (precision + recall)

    f1_total += f1

em = em_count / len(test_b)
f1_avg = f1_total / len(test_b)

print(f"Exact Match: {em:.4f}")
print(f"Token F1: {f1_avg:.4f}")

Exact Match: 0.5833
Token F1: 0.8722


In [ ]:
for i in range(min(10, len(test_b))):
    pred = predicted_answers.get(i, "")
    truth = test_b[i]["answer_text"]
    print(f"[{i}] pred={pred!r} | truth={truth!r}")

[0] pred='OAuth2' | truth='OAuth2'
[1] pred='Docker Compose' | truth='Docker Compose'
[2] pred='16' | truth='PostgreSQL 16'
[3] pred='valid Bearer token' | truth='Bearer token'
[4] pred='pool_timeout error' | truth='pool_timeout'
[5] pred='CoreDNS' | truth='CoreDNS'
[6] pred='2 AM' | truth='2 AM'
[7] pred='60 minutes' | truth='60 minutes'
[8] pred='SFTTrainer class' | truth='SFTTrainer'
[9] pred='zero' | truth='zero downtime'


In [ ]:
print(trainer_b.state.best_model_checkpoint)
print(trainer_b.state.best_metric)

models/intent_classifier/checkpoint-28
0.17058777809143066


# Model C

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import is_bitsandbytes_available
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

MODEL_C = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer_c = AutoTokenizer.from_pretrained(MODEL_C)
if tokenizer_c.pad_token is None:
    tokenizer_c.pad_token = tokenizer_c.eos_token

In [ ]:
# Refresh the cached check if bitsandbytes was installed after the kernel started.
is_bitsandbytes_available.cache_clear()
use_qlora = torch.cuda.is_available()

if use_qlora:
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    base_c = AutoModelForCausalLM.from_pretrained(
        MODEL_C,
        quantization_config=quant_config,
        device_map="auto",
    )
    base_c = prepare_model_for_kbit_training(base_c)
else:
    base_c = AutoModelForCausalLM.from_pretrained(MODEL_C)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model_c = get_peft_model(base_c, lora_config)
model_c.print_trainable_parameters()

Loading weights: 100%|██████████| 398/398 [00:02<00:00, 147.94it/s]


trainable params: 2,949,120 || all params: 4,025,417,216 || trainable%: 0.0733


In [ ]:
from model_c_eval import encode_conversation, completion_metrics, generate_answer

# Pre-tokenize and mask every prompt token; only assistant answers carry loss.
def format_for_sft(example):
    return encode_conversation(tokenizer_c, example['messages'])


In [ ]:
import json
from pathlib import Path
from datasets import Dataset, DatasetDict

# Curated lab conversations: troubleshooting, uncertainty, tool-result synthesis,
# and escalation. Splits are fixed and independent of Model B's QA dataset.
sft_records = [
    json.loads(line)
    for line in Path('data/support_sft.jsonl').read_text().splitlines()
    if line.strip()
]
sft_dataset = DatasetDict({
    split: Dataset.from_list([
        {'messages': record['messages']}
        for record in sft_records if record['split'] == split
    ])
    for split in ['train', 'validation', 'test']
})
sft_train = sft_dataset['train'].map(format_for_sft, remove_columns=['messages'])
sft_val = sft_dataset['validation'].map(format_for_sft, remove_columns=['messages'])
sft_test = sft_dataset['test'].map(format_for_sft, remove_columns=['messages'])
print(sft_dataset)

# Keep the held-out Golden Set separate from training and validation.
golden_set = [json.loads(line) for line in Path('data/golden_set.jsonl').read_text().splitlines() if line.strip()]
assert not {case['prompt'] for case in golden_set} & {
    r['messages'][1]['content'] for r in sft_records
}
assert all(any(label != -100 for label in row['labels']) for row in sft_train)


Map: 100%|██████████| 8/8 [00:00<00:00, 1698.44 examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 60
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 8
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 8
    })
})


In [ ]:
from datetime import datetime, timezone
from transformers import EarlyStoppingCallback, set_seed
import hashlib

# Run the Model C loading + LoRA cells first to create a fresh adapter.
# LoRA B weights start at zero; reject accidental continuation of a trained adapter.
if any(torch.count_nonzero(p).item() for name, p in model_c.named_parameters() if 'lora_B' in name):
    raise RuntimeError('Reload Model C and create a fresh LoRA adapter before this run.')
set_seed(42)
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
run_dir = Path('models') / ('support_adapter_' + run_id)
eval_dir = Path('results/model_c_evaluation') / run_id
eval_dir.mkdir(parents=True, exist_ok=False)
system_message = sft_records[0]['messages'][0]
manifest = {
    'model': MODEL_C, 'run_id': run_id, 'seed': 42,
    'quantized_4bit': use_qlora, 'loss_scope': 'assistant completion only',
    'max_new_tokens': 384, 'do_sample': False,
    'data_sha256': hashlib.sha256(Path('data/support_sft.jsonl').read_bytes()).hexdigest(),
    'golden_sha256': hashlib.sha256(Path('data/golden_set.jsonl').read_bytes()).hexdigest(),
    'note': 'Golden Set is a known regression suite, not an unseen final benchmark.',
}
(eval_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))

# Record a real baseline before any optimization. Keep incremental outputs.
baseline_metrics = completion_metrics(model_c, tokenizer_c, sft_dataset['test'], baseline=True)
(eval_dir / 'baseline_metrics.json').write_text(json.dumps(baseline_metrics, indent=2))
baseline_answers = {}
for case in golden_set:
    baseline_answers[case['id']] = generate_answer(
        model_c, tokenizer_c, [system_message, {'role': 'user', 'content': case['prompt']}], baseline=True)
    (eval_dir / 'baseline_answers.json').write_text(json.dumps(baseline_answers, indent=2))
    print(case['id'], 'baseline recorded')

sft_args = SFTConfig(
    output_dir=str(run_dir), num_train_epochs=4,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=4, learning_rate=2e-5,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    max_length=512, packing=False, report_to='none', seed=42, data_seed=42,
    bf16=bool(use_qlora and torch.cuda.is_bf16_supported()),
    fp16=bool(use_qlora and not torch.cuda.is_bf16_supported()),
    gradient_checkpointing=True, logging_steps=5,
)
trainer_c = SFTTrainer(
    model=model_c, args=sft_args, train_dataset=sft_train, eval_dataset=sft_val,
    processing_class=tokenizer_c,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
# Verify the actual trainer retains the assistant-only labels.
assert trainer_c.train_dataset[0]['labels'] == sft_train[0]['labels']
trainer_c.train()
trainer_c.save_model(str(run_dir / 'final'))
tokenizer_c.save_pretrained(str(run_dir / 'final'))
(eval_dir / 'training_history.json').write_text(json.dumps(trainer_c.state.log_history, indent=2))
print('Saved new run:', run_dir, eval_dir)


G01 baseline recorded
G02 baseline recorded
G03 baseline recorded
G04 baseline recorded
G05 baseline recorded
G06 baseline recorded
G07 baseline recorded
G08 baseline recorded
G09 baseline recorded
G10 baseline recorded


Truncating train dataset: 100%|██████████| 60/60 [00:00<00:00, 13803.11 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 60/60 [00:00<00:00, 31049.75 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 8/8 [00:00<00:00, 6049.11 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,5.293569,5.169939,1.264885,6318.000000,0.300711
2,4.792657,4.827332,1.371220,12636.000000,0.315051
3,4.287396,4.639854,1.433252,18954.000000,0.321092
4,4.450264,4.583216,1.449605,25272.000000,0.317714


Saved new run: models/support_adapter_20260919T141721966045 results/model_c_evaluation/20260919T141721966045


# Phase B — Model C evaluation and quality gates

In [ ]:
if 'trainer_c' not in globals() or trainer_c.state.global_step == 0:
    raise RuntimeError('Run the new baseline and training cell first.')
eval_model_c = trainer_c.model
model_report = {
    'manifest': manifest,
    'baseline_method': 'Recorded before training with disabled adapter',
    'split': 'test', 'loss_scope': 'assistant completion only, token-weighted',
    'baseline': baseline_metrics,
    'fine_tuned': completion_metrics(eval_model_c, tokenizer_c, sft_dataset['test']),
}
model_report['quality_gate'] = {
    'loss_improved': model_report['fine_tuned']['loss'] < model_report['baseline']['loss'],
    'golden_set': 'pending human review', 'accepted': False,
}
(eval_dir / 'metrics.json').write_text(json.dumps(model_report, indent=2, allow_nan=False))
print(json.dumps(model_report, indent=2))


{
  "manifest": {
    "model": "Qwen/Qwen3-4B-Instruct-2507",
    "run_id": "20260919T141721966045",
    "seed": 42,
    "quantized_4bit": true,
    "loss_scope": "assistant completion only",
    "max_new_tokens": 384,
    "do_sample": false,
    "data_sha256": "1c583da91963a8a53360367c2935f895720f7da2d5fad2f697893f589218cc53",
    "golden_sha256": "c3f3251297baf134e4ed46d544829f048ccda0c8d64599fdb94a6d02a5d71a00",
    "note": "Golden Set is a known regression suite, not an unseen final benchmark."
  },
  "baseline_method": "Recorded before training with disabled adapter",
  "split": "test",
  "loss_scope": "assistant completion only, token-weighted",
  "baseline": {
    "loss": 5.361577132652545,
    "perplexity": 213.0607066081621,
    "target_tokens": 290
  },
  "fine_tuned": {
    "loss": 4.414613251850523,
    "perplexity": 82.64987003987758,
    "target_tokens": 290
  },
  "quality_gate": {
    "loss_improved": true,
    "golden_set": "pending human review",
    "accepted": false

## Golden Set: generate both versions, then review

These 10 cases are separate from the SFT data. Generation is deterministic. Judge each response against its rubric: keyword matches alone cannot establish groundedness or safe escalation. Review files retain both responses and start with null verdicts; no case passes automatically. Generation may take several minutes on CPU.


In [ ]:
review_path = eval_dir / ('golden_review_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f') + '.json')
reviews = []
for case in golden_set:
    baseline = baseline_answers[case['id']]
    tuned = generate_answer(eval_model_c, tokenizer_c,
        [system_message, {'role': 'user', 'content': case['prompt']}])
    reviews.append({**case,
        'baseline_response': baseline['text'], 'fine_tuned_response': tuned['text'],
        'baseline_generation': {k: v for k, v in baseline.items() if k != 'text'},
        'fine_tuned_generation': {k: v for k, v in tuned.items() if k != 'text'},
        'baseline_pass': None, 'fine_tuned_pass': None,
        'error_category': '', 'review_notes': '',
    })
    review_path.write_text(json.dumps(reviews, indent=2))
    print(case['id'], 'truncated:', baseline['truncated'], tuned['truncated'])
print('Review file:', review_path)


G01 truncated: False False
G02 truncated: True True
G03 truncated: False False
G04 truncated: False False
G05 truncated: True True
G06 truncated: False False
G07 truncated: False False
G08 truncated: False False
G09 truncated: True True
G10 truncated: True True
Review file: results/model_c_evaluation/20260919T141721966045/golden_review_20260919T142038435976.json


## Apply the gate after review

In the generated review JSON, set `baseline_pass` and `fine_tuned_pass` to `true` or `false` and explain the verdict in `review_notes`. For failures, record an `error_category` such as hallucination, missed escalation, missing clarification, or instruction following. Then run the cell below. Acceptance requires lower test loss, all required fine-tuned cases passing, and completed reviews. This is Model C's gate only, not a system deployment gate.


In [ ]:
reviews = json.loads(review_path.read_text())
expected = {case['id'] for case in golden_set}
if len(reviews) != len(expected) or {row['id'] for row in reviews} != expected:
    raise ValueError('Review must contain each Golden Set case exactly once.')
required_ids = {case['id'] for case in golden_set if case['required']}
complete = all(
    type(row['baseline_pass']) is bool and type(row['fine_tuned_pass']) is bool
    and bool(row['review_notes'].strip()) for row in reviews
)
regressions = [row['id'] for row in reviews
               if row['id'] in required_ids and row['baseline_pass'] is True
               and row['fine_tuned_pass'] is False]
truncated_cases = [row['id'] for row in reviews if any(
    row[key]['truncated'] for key in ['baseline_generation', 'fine_tuned_generation'])]
required_pass = complete and all(row['fine_tuned_pass'] is True
                               for row in reviews if row['id'] in required_ids)
model_report['quality_gate'].update({
    'golden_set': 'reviewed' if complete else 'pending human review',
    'required_cases_pass': required_pass,
    'regressions': regressions,
    'truncated_cases': truncated_cases,
    'review_file': str(review_path),
    'accepted': bool(model_report['quality_gate']['loss_improved'] and required_pass and not regressions and not truncated_cases),
})
(eval_dir / 'metrics.json').write_text(json.dumps(model_report, indent=2, allow_nan=False))
print(json.dumps(model_report['quality_gate'], indent=2))


{
  "loss_improved": true,
  "golden_set": "pending human review",
  "accepted": false,
  "required_cases_pass": false,
  "regressions": [],
  "truncated_cases": [
    "G02",
    "G05",
    "G09",
    "G10"
  ],
  "review_file": "results/model_c_evaluation/20260919T141721966045/golden_review_20260919T142038435976.json"
}


## Diagnose the current adapter and compare concise responses
Run the next cell directly while the trained model remains in memory. It does not train or replace previous results. New training examples only take effect after rerunning data preparation and a fresh training run.


In [ ]:
# Run only this cell on the current trained model; no retraining required.
import importlib
import model_c_eval
importlib.reload(model_c_eval)
from model_c_eval import adapter_diagnostics, generate_answer

if 'trainer_c' not in globals() or trainer_c.state.global_step == 0:
    raise RuntimeError('This cell needs the trained trainer_c from the current run.')
current_model = trainer_c.model
probe_messages = [sft_records[0]['messages'][0], {
    'role': 'user', 'content': 'A background task is delayed. What information should I collect first?'}]
diagnostics = adapter_diagnostics(current_model, tokenizer_c, probe_messages)
print(json.dumps(diagnostics, indent=2))
comparison_dir = eval_dir / ('concise_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
comparison_dir.mkdir(parents=True, exist_ok=False)
(comparison_dir / 'adapter_diagnostics.json').write_text(json.dumps(diagnostics, indent=2))
if not diagnostics['effect_detected']:
    raise RuntimeError('No adapter effect on this probe. Investigate before retraining or scoring.')

# Same instruction and token budget for both models. This is a new prompting
# experiment, not directly comparable to the earlier prompting configuration.
concise_system = dict(sft_records[0]['messages'][0])
concise_system['content'] += (
    ' Answer directly in at most 100 words. Put the decision and essential next '
    'action first. Avoid preambles, repeated explanations, and speculative advice.'
)
(comparison_dir / 'generation_config.json').write_text(json.dumps({
    'system_message': concise_system, 'max_new_tokens': 384, 'do_sample': False,
    'model': MODEL_C, 'adapter_run': str(run_dir),
    'baseline_method': 'Current base with adapter disabled; prompting experiment after training',
}, indent=2))
concise_reviews = []
for case in golden_set:
    messages = [concise_system, {'role': 'user', 'content': case['prompt']}]
    base = generate_answer(current_model, tokenizer_c, messages, baseline=True)
    tuned = generate_answer(current_model, tokenizer_c, messages)
    concise_reviews.append({**case,
        'baseline_response': base['text'], 'fine_tuned_response': tuned['text'],
        'baseline_generation': {k:v for k,v in base.items() if k != 'text'},
        'fine_tuned_generation': {k:v for k,v in tuned.items() if k != 'text'},
        'baseline_pass': None, 'fine_tuned_pass': None, 'error_category': '', 'review_notes': '',
    })
    (comparison_dir / 'review.json').write_text(json.dumps(concise_reviews, indent=2))
    print(case['id'], base['finish_reason'], tuned['finish_reason'])
print('Review the new experiment separately:', comparison_dir / 'review.json')


{
  "active_adapters": [
    "default"
  ],
  "adapter_layers": 72,
  "lora_b_nonzero_elements": 1474560,
  "max_logit_difference": 4.40625,
  "mean_logit_difference": 0.8963900804519653,
  "effect_detected": true,
  "note": "Nonzero B weights are evidence of updates for this zero-B initialization. Logit differences establish an effect on this probe, not better behavior."
}
G01 eos eos
G02 eos eos
G03 eos eos
G04 eos eos
G05 eos eos
G06 eos eos
G07 eos eos
G08 eos eos
G09 eos eos
G10 eos eos
Review the new experiment separately: results/model_c_evaluation/20260919T141721966045/concise_20260919T142300568080/review.json
